# Backtranslation
Backtranslation didn't work as intended in the data_augmentation.ipynb notebook. Therefore this notebook tries BT solo, with different approaches.
The placeholder "MBTITOKENPLACEHOLDER" used in the initial approach didn't work. This is most likely due to two things:
- Too many tokens: the word is too long, is broken down into too many tokens, of which some might get translated resulting in typos when translated back
- Related to the other point, the placeholder containing the words "Token" and "Placeholder" might lead to translating back and forth, while the goal of the placeholder is to prevent this from happening

## First approach
The first approach uses a short english name ("John") as a placeholder. A name like "John" won't get translated, and is processed as only one token. Furthermore, it can substitute <MBTI> since both of them would be in the third person singular.

In [ ]:
from transformers import MarianMTModel, MarianTokenizer
import numpy as np
import pandas as pd
import torch
import gc
from tqdm import tqdm
import re

tqdm.pandas()


rseed = 42

In [ ]:
from huggingface_hub import login
from getpass import getpass

hftoken = getpass("HF-Token ")
login(token=hftoken)

In [ ]:
PLACEHOLDER = "John"

def protect_mbti_mask(text):
    return text.replace("<mbti>", PLACEHOLDER)

def restore_mbti_mask(text):
    return re.sub(rf"{PLACEHOLDER}", "<mbti>", text, flags=re.IGNORECASE)

In [ ]:
def backtranslate_safe(texts, device="cuda"):
    en_de_tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")
    en_de_mod = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-de").to(device)
    de_en_tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-de-en")
    de_en_mod = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-de-en").to(device)

    gen_kwargs = dict(
        num_beams=4,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        max_length=512,
        early_stopping=True,
    )

    def translate(model, tokenizer, batch):
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                          truncation=True, max_length=512).to(device)
        with torch.no_grad():
            out = model.generate(**inputs, **gen_kwargs)
        return tokenizer.batch_decode(out, skip_special_tokens=True)

    protected = [protect_mbti_mask(t) for t in texts]
    results = []
    for i in range(0, len(protected), 16):
        batch = protected[i:i+16]
        de = translate(en_de_mod, en_de_tok, batch)
        back = translate(de_en_mod, de_en_tok, de)
        results.extend(back)

    del en_de_mod, de_en_mod
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return [restore_mbti_mask(t) for t in results]

In [ ]:
# read data
bt_sample = pd.read_csv("~/MA/data/bt_augmented.csv")

In [ ]:
texts = bt_sample["post"].tolist()
bt_df = bt_sample["post_augmented_john"] = backtranslate_safe(texts)

In [ ]:
bt_df.to_csv("bt_augmented_john.csv", sep='\t', index=False, quoting=1)